# Upper-bound validation walkthrough — one metric, end to end

**Metric:** `aigner_ziegler_proofs_from_the_book` — the *Proofs from the Book* standard for mathematical
proofs (elegance / intelligibility), turned into an LLM-as-judge rubric and evolved by **GEPA**.

This notebook lets you **personally validate the whole pipeline** on this single metric. Nothing is
hard-coded: every number is recomputed from the real GEPA registry and the real scored arrays
(staged under `data/aigner_validation/`). It walks through:

1. **The metric in full** — the human seed(s) and the final GEPA-evolved rubric.
2. **How GEPA runs** — the loop, the real prompts, a diagram, and this metric's actual lineage.
3. **Ω** — the atomic units mined from *all* evolutions (and a schema bug manual inspection caught).
4. **The input→label data** — real corpus items, and how M and each Ω criterion were scored.
5. **The math** — exactly how we bound recoverability, with every estimator inlined so you read it
   rather than trust an import.
6. **The upper bound, stated** — and honestly, what kind of bound it is.

> Run order matters; run top to bottom. Zero GPU. Pure `numpy` + `pandas` + `scikit-learn`.

## 0. Notation & what each measure means — read this first

Three judge outputs, all from the **same** weak judge on the **same** item — differing only in what it's asked:

| symbol | the prompt the judge gets | output |
|---|---|---|
| **M** (holistic target) | *"Overall, is this answer high quality — top-tier to a strong expert?"* — ONE all-things-considered question, NOT decomposed | one `P(YES)` → median-split bit |
| **Xᵢ** (per-criterion signal) | criterion `eᵢ` **alone** as the rubric | one `P(YES)` per criterion; keep all K as a **vector** |
| **M̂_S** (executor verdict) | the criteria in subset `S` listed **together** in one rubric | **one collapsed** `P(YES)` |

- **M is NOT the GEPA rubric's output.** It's a deliberately *generic, decoupled* "is this good overall?"
  judgment — the thing we try to reconstruct. We decouple it from the criteria on purpose: if M were the
  rubric's own aggregate, the criteria would predict it trivially (circular). *Caveat:* this makes M a
  generic-quality proxy, **not** the specific "elegance" intent and **not** ground truth. A defensible
  alternative is `M = the full-rubric verdict`; the run chose holistic to avoid circularity.
- **Xᵢ vs M̂_S is the whole game.** `{Xᵢ}_{i∈S}` is **K separate numbers** (a vector); `M̂_S` is **one
  number** the judge produces after reading all of S at once. The *executor bottleneck* is exactly that
  collapse of K → 1.

**Two recovery measures — different jobs, never subtracted:**

- **Ǐ(S) = I(M; X_S)** — Shannon MI between the holistic target and the *vector* of per-criterion signals.
  This is **SUFFICIENCY / reconstructability**: *if you knew all K criterion answers, how well could you
  predict M?* High ⇒ the criteria capture the judgment (**articulable**); low ⇒ M has content the criteria
  miss (**tacit/taste**). That **is** the articulability question — M = the thing to explain, X_S = the
  explanation. **It is NOT consistency.**
    - *Consistency* (re-run M, perturb it, compare passes) is a **different axis** = reliability/stability.
      We handle it separately (test-retest, the continuous-`P(YES)` readout, GEPA's reliability term), and
      it *upper-bounds* recovery (you can't rebuild an unstable target). The multi-pass
      `I_V = H(p̄) − mean_i H(p_i)` is the consistency-flavored cousin; this notebook uses sufficiency.
- **R(S) = I_TVD(M; M̂_S)** — recovery through the *executor*: how well the single collapsed verdict tracks
  M, in **total-variation** MI: `R = 2·π·(1−π)·|p̄₁ − p̄₀|`, π=P(M=1), p̄_c = mean verdict on M=c items.

**Why TVD and not Shannon for R?** It IS the same MI with a different f (TV: `f(t)=½|t−1|`; Shannon:
`f(t)=t·log t`). We use TVD for the *recovery* layer for four concrete reasons:
1. **Bounded, with an operational cap.** TVD-MI ∈ [0, ½] for a binary verdict, and ½ *means* "the most one
   binary verdict can transmit." Our cap `1−1/min(N,K)` is clean in TVD; Shannon's 1-bit cap tangles with base rate.
2. **Estimable from the continuous verdict.** `2π(1−π)|p̄₁−p̄₀|` is a smooth plug-in over `P(YES)` — no
   joint-distribution binning. Shannon joint-MI needs the full K-way joint (undersampled at N≈200 — that's
   why exact-γ is untrusted).
3. **Decision-relevance.** TV bounds how much knowing M̂ can change the expected loss of *any* bounded
   decision about M (`|E_P g − E_Q g| ≤ 2·TV·‖g‖∞`). A metric exists to drive a decision, so TV is the right
   divergence; Shannon bits measure *compressibility*. (`|p̄₁−p̄₀|` is literally the verdict's class-separation.)
4. **Same-f ladder + DPI.** Transmission/recovery/articulability all in TVD ⇒ gaps are comparable
   *transmission losses*, and data-processing holds termwise (the K→1 collapse can only lose TVD-info).

**Why we still keep Shannon:** TVD has no chain rule, so the *submodularity / discovery* layer (γ,
greedy-vs-OPT, the co-information identity) is governed by **Shannon** — that's why `Ǐ` is Shannon.
So: **Shannon for selection structure, TVD for transmission.** Different questions; never differenced.

In [1]:
import ast, glob, itertools, json, os, re
import numpy as np

# resolve the staged-data dir whether you run from repo root or from notebooks/
CAND = ["notebooks/data/aigner_validation", "data/aigner_validation", "./aigner_validation"]
ROOT = next((p for p in CAND if os.path.isdir(p)), None)
assert ROOT, "could not find aigner_validation data dir"
REG = f"{ROOT}/registry"
NPZ = f"{ROOT}/npz"
FAM = "claude-parsed__aigner_ziegler_proofs_from_the_book.md"
RUNS = sorted(glob.glob(f"{REG}/{FAM}_*"))
print("data root:", ROOT)
print("GEPA runs found:", [os.path.basename(r) for r in RUNS])


def load_body(vf):
    """Return (full_json, parsed_body). body is a free-text string (seed) or a dict (structured rubric)."""
    o = json.load(open(vf))
    b = o.get("body")
    if isinstance(b, dict):
        return o, b
    for L in (json.loads, ast.literal_eval):
        try:
            return o, L(b)
        except Exception:
            pass
    return o, b

data root: data/aigner_validation
GEPA runs found: ['claude-parsed__aigner_ziegler_proofs_from_the_book.md_0', 'claude-parsed__aigner_ziegler_proofs_from_the_book.md_1', 'claude-parsed__aigner_ziegler_proofs_from_the_book.md_2']


## 1. The metric in full

GEPA was seeded **three independent times** from three different human statements of the
Aigner–Ziegler standard, then evolved each into a structured rubric. Below: each run's seed, and the
final evolved rubric of run 0 (the one whose criteria were actually scored downstream).

Watch for two things: (a) the three seeds emphasize **different facets** (undergraduate-intelligibility,
self-containment, and *elegance / non-ugliness*); (b) the final rubric **drifts** — its title mentions
"competitive programming" even though the metric is about math proofs.

In [2]:
def render_rubric(b, indent=""):
    if isinstance(b, str):
        return b
    out = [f"{indent}NAME: {b.get('name','')}", f"{indent}DESC: {b.get('description','')}"]
    for i, c in enumerate(b.get("criteria", []) or []):
        if not isinstance(c, dict):
            out.append(f"{indent}  [{i}] {c}"); continue
        out.append(f"{indent}  [{i}] {c.get('name','')}: {c.get('description','')}")
        for ch in (c.get("checks", []) or []):
            if isinstance(ch, dict):
                out.append(f"{indent}      - check: {ch.get('name','')}: {ch.get('description','')}")
                for s in (ch.get("steps", []) or []):
                    out.append(f"{indent}          . step: {s}")
            else:
                out.append(f"{indent}      - check: {ch}")
    # alternate schema: criteria stored as top-level named dimensions
    named = {k: v for k, v in b.items()
             if isinstance(v, dict) and isinstance(v.get("description"), str)
             and k not in ("scores", "scoring")}
    for k, v in named.items():
        out.append(f"{indent}  <{k}>: {v.get('description','')}")
    return "\n".join(out)


print("==== THE THREE GEPA SEEDS (v000 of each run) ====")
for r in RUNS:
    _, b0 = load_body(sorted(glob.glob(f"{r}/v*__prompt.json"))[0])
    print(f"\n[{os.path.basename(r)[-1]}] {render_rubric(b0)[:400]}")

print("\n\n==== FINAL EVOLVED RUBRIC (run 0, last version) ====")
vfs0 = sorted(glob.glob(f"{RUNS[0]}/v*__prompt.json"))
_, bfin = load_body(vfs0[-1])
print(render_rubric(bfin))

==== THE THREE GEPA SEEDS (v000 of each run) ====

[0] Every proof must be intelligible to someone with basic undergraduate mathematics.

Scoring guidance: Select proofs that do not presuppose advanced graduate-level background; aim for arguments a competent undergraduate reader can follow. This is a stated operational constraint by Aigner & Ziegler.

[1] The proof should be readable without extensive prerequisite reading.

Scoring guidance: Avoid proofs that require consulting many external results or long chains of prior theory. The proof should stand on its own within a short chapter or section.

[2] Reject proofs that are 'ugly' — i.e., rely on brute force or inelegant calculations without conceptual clarity.

Scoring guidance: Aigner & Ziegler invoke Hardy's aesthetic standard: permanent ugliness disqualifies a proof from the Book. Prefer elegance and conceptual neatness over mere correctness.


==== FINAL EVOLVED RUBRIC (run 0, last version) ====
NAME: Automated Evaluator Rubric 

## 2. How GEPA runs — the loop, the prompts, and this metric's lineage

GEPA (Genetic-Pareto reflective prompt evolution) does **not** hand-write criteria. It starts from the
human seed and repeatedly: *executes* the rubric with a **weak** judge, *measures* where it fails,
asks a **strong** reviser to *reflect* on those failures and apply **one mutation operator**, then
*selects* the best survivors. No ground-truth labels are ever used — the objective is **fidelity**
(does the rubric encode its intent reliably and recoverably).

```
   human seed (v000)                                    ┌───────────────────────────┐
   "every proof must be intelligible…"        ┌────────►│  WEAK judge model         │
            │ INIT                            │         │  executes the rubric in   │
            ▼                                 │         │  [0,1], one item at a time│
     ┌──────────────┐   score (NO labels):    │         └───────────────────────────┘
     │  candidate   │── fidelity = reconstruction + counterfactual +
     │   rubric     │   reliability + consistency   ◄──────────────┘
     └──────────────┘            │
            ▲                     ▼
            │            ┌─────────────────────┐   STRONG reviser tags each failure:
            │            │  attribute failures │   AMBIGUOUS_PROMPT  (rubric's fault)
            │            └─────────────────────┘   JUDGE_LIMITATION  (weak reader can't apply)
            │                     │
            │                     ▼
            │            ┌─────────────────────┐   pick ONE operator + rewrite:
            │            │   reflect / mutate  │   CLARIFY · ANCHOR · EDGE   ← AMBIGUOUS_PROMPT
            │            └─────────────────────┘   DECOMPOSE · MECHANIZE     ← JUDGE_LIMITATION
            │                     │                 FEWSHOT+ · PRUNE
            │                     ▼
            │            ┌─────────────────────┐
            │            │   budget filter     │  reject if > token / fewshot / AST cap
            │            └─────────────────────┘
            │                     │
            │                     ▼
            │   ┌──────────────────────────────────────┐
            └───│ truncation-select pool (keep top-k by │  best (promotable-first, then fidelity)
                │ fidelity) → becomes next round's parent│  → repeat for R rounds
                └──────────────────────────────────────┘
                                  │
                                  ▼
                 acceptance test on FRESH cross-family roles → promote to HEAD only if > seed
```

**Why this matters for us.** The structured `criteria → checks → steps` we mine into Ω are the direct
**product of the `DECOMPOSE` + `MECHANIZE` operators** firing on `JUDGE_LIMITATION` failures — i.e.
GEPA is literally *trying to articulate tacit judgment into procedure a weak reader can follow.* Ω is
that articulation. (Also: the revise prompt below is templated to *"competitive-programming solution"* —
which is exactly why §1's evolved math-proof rubric drifted to that wording. The drift is baked into
the operator prompt, not random.)

### The two real GEPA prompts (verbatim from `methods/metric_implementer/optimizer.py`)

**(1) Failure attribution** — routes which operator is even allowed:
```
A rubric is executed by a WEAKER model than you. For each failure below, decide: could a careful but
limited reader score it correctly from the rubric alone (then the failure is the rubric's fault:
AMBIGUOUS_PROMPT), or does correct scoring require judgment the rubric cannot transmit to a weak reader
as written (JUDGE_LIMITATION)?

RUBRIC: {body}
FAILURES: {failures}
Respond ONLY with JSON: {"attributions": ["AMBIGUOUS_PROMPT"|"JUDGE_LIMITATION", ...]}
```

**(2) Reflective mutation** — reads measured failures, picks ONE operator, rewrites the rubric:
```
You are improving the RUBRIC of an automated evaluator (an LLM judge that scores one
competitive-programming solution at a time in [0,1]).

CURRENT RUBRIC: {body}
MEASURED PROBLEMS (from validity testing — real failures, not hypotheticals): {failures}

Produce ONE revised rubric that fixes the most damaging problem. Pick exactly one mutation operator:
- CLARIFY:   sharpen the definition against the confusion named above
- FEWSHOT+:  add ONE worked example (use a failed counterfactual pair if shown)
- ANCHOR:    define precisely what scores of 0.0, 0.5, 1.0 mean
- EDGE:      add an explicit edge-case clause for an item the judge flip-flopped on
- PRUNE:     remove text that adds instability without adding meaning
- DECOMPOSE: split into 2-3 sub-questions plus an explicit aggregation rule
- MECHANIZE: rewrite judgment-language as explicit step-by-step checks a much weaker reader
             could follow mechanically, with an explicit aggregation rule

failures tagged JUDGE_LIMITATION need DECOMPOSE or MECHANIZE (transmit less judgment, more procedure);
failures tagged AMBIGUOUS_PROMPT need CLARIFY, ANCHOR, or EDGE.
HARD CONSTRAINT: stay within {token_cap} tokens and at most {fewshot_cap} worked examples.
Respond ONLY with JSON: {"operator": "<one of {ops}>", "rubric": "<full revised rubric>",
                         "rationale": "<one sentence>"}
```

The full operator menu is `INIT, CLARIFY, FEWSHOT+, ANCHOR, EDGE, PRUNE, DECOMPOSE, MECHANIZE`.

In [3]:
# The ACTUAL operator sequence GEPA applied to this metric (real registry lineage, not the template)
from collections import Counter
print("Operators fired across the 3 GEPA runs:")
allops = Counter()
for vf in sorted(glob.glob(f"{REG}/{FAM}_*/v*__prompt.json")):
    allops[(json.load(open(vf)).get("lineage") or {}).get("operator")] += 1
print("  ", dict(allops), "\n")

for r in RUNS:
    print(f"=== {os.path.basename(r)} ===")
    for vf in sorted(glob.glob(f"{r}/v*__prompt.json")):
        o = json.load(open(vf)); lin = o.get("lineage") or {}
        vid = os.path.basename(vf).split("__")[0]
        rat = (o.get("notes") or "").replace("reviser_rationale:", "").strip()
        print(f"  {vid}  round={lin.get('optimizer_round')}  op={str(lin.get('operator')):<10}"
              f" parent={lin.get('parent_version')}")
        if rat:
            print(f"          rationale: {rat[:120]}")

Operators fired across the 3 GEPA runs:
   {'INIT': 6, 'MECHANIZE': 23, 'DECOMPOSE': 7} 

=== claude-parsed__aigner_ziegler_proofs_from_the_book.md_0 ===
  v000  round=0  op=INIT       parent=None
  v001  round=1  op=MECHANIZE  parent=v000
          rationale: This revised rubric breaks down the evaluation into explicit, step-by-step checks to make it more mechanically executabl
  v002  round=1  op=MECHANIZE  parent=v000
          rationale: This revised rubric breaks down the evaluation into explicit, step-by-step checks to make it more mechanically executabl
  v003  round=1  op=MECHANIZE  parent=v000
          rationale: This revised rubric breaks down the evaluation into explicit, step-by-step checks to make it more mechanically executabl
  v004  round=2  op=MECHANIZE  parent=v000
          rationale: This revised rubric breaks down the evaluation into explicit, step-by-step checks to make it more mechanically executabl
  v005  round=2  op=MECHANIZE  parent=v000
          rationale:

### Every version, compactly

Now the lineage as data. Three tags appear — and the confusing ones are real, not bugs:

- **`[seed/text]`** — a free-text body. This is a human seed. It appears at **v000** *and again at v007*
  because run 0 actually concatenates **two** `improve()` passes: the first (v000→v006) degenerated, so
  GEPA **re-seeded** with a fresh `INIT` at v007 and tried again (v008→v013).
- **`[structured] (empty)`** — the body parsed as JSON but held **no minable criteria**. For v001–v006
  the body is a bare *score dict* like `{"legibility":0.0,"mathematical_depth":0.5,"clarity":1.0}` — an
  early mutation that emitted a **score output instead of a rubric**. Nothing to mine.
- **`[structured] ['Intelligibility','Validity']`** — a real evolved rubric with named criteria.

So the "messy" middle is GEPA flailing (emitting scores, not rubrics) until the re-seed at v007 finally
produced usable rubrics. The cell after the listing dumps the actual bodies so you can see it.

In [4]:
for r in RUNS:
    vfs = sorted(glob.glob(f"{r}/v*__prompt.json"))
    print(f"\n=== run {os.path.basename(r)}  ({len(vfs)} versions) ===")
    for vf in vfs:
        o, b = load_body(vf)
        vid = os.path.basename(vf).split("__")[0]
        if isinstance(b, str):
            print(f"  {vid}: [seed/text]   {b[:78].replace(chr(10),' ')}")
        elif isinstance(b, dict):
            names = [c.get("name", "?") for c in (b.get("criteria") or []) if isinstance(c, dict)]
            named = [k for k, v in b.items() if isinstance(v, dict) and isinstance(v.get("description"), str)
                     and k not in ("scores", "scoring")]
            tag = (names or named) or "(empty)"
            print(f"  {vid}: [structured]  {tag}")
        else:
            print(f"  {vid}: [unparsed]")


=== run claude-parsed__aigner_ziegler_proofs_from_the_book.md_0  (14 versions) ===
  v000: [seed/text]   Every proof must be intelligible to someone with basic undergraduate mathemati
  v001: [structured]  (empty)
  v002: [structured]  (empty)
  v003: [structured]  (empty)
  v004: [structured]  (empty)
  v005: [structured]  (empty)
  v006: [structured]  (empty)
  v007: [seed/text]   Every proof must be intelligible to someone with basic undergraduate mathemati
  v008: [structured]  ['Intelligibility', 'Validity']
  v009: [structured]  ['Intelligibility', 'Graduate-Level Background']
  v010: [structured]  ['Intelligibility', 'Graduate-Level Background']
  v011: [structured]  ['Intelligibility', 'Validity']
  v012: [structured]  ['Intelligibility', 'Validity']
  v013: [structured]  ['Intelligibility', 'Validity']

=== run claude-parsed__aigner_ziegler_proofs_from_the_book.md_1  (8 versions) ===
  v000: [seed/text]   The proof should be readable without extensive prerequisite reading.  S

In [5]:
# concretely: an "(empty)" version vs a real one (run 0)
R0 = f"{REG}/{FAM}_0"
_, b_empty = load_body(f"{R0}/v001__prompt.json")
_, b_real = load_body(f"{R0}/v008__prompt.json")
print("v001 body  (rendered '(empty)') =", json.dumps(b_empty))
print("   -> a bare SCORE dict, not a rubric: keys are dimensions, values are 0.0/0.5/1.0 scores.\n")
print("v008 body  (rendered with criteria) keys =", list(b_real.keys()))
print("   criteria:", [c.get("name") for c in b_real.get("criteria", []) if isinstance(c, dict)])

v001 body  (rendered '(empty)') = {"legibility": 0.0, "mathematical_depth": 0.5, "clarity": 1.0}
   -> a bare SCORE dict, not a rubric: keys are dimensions, values are 0.0/0.5/1.0 scores.

v008 body  (rendered with criteria) keys = ['name', 'description', 'criteria', 'worked_examples', 'scores']
   criteria: ['Intelligibility', 'Validity']


### What a real GEPA mutation actually output — full text, not a summary

The previous cells *summarized* versions (criterion names). Here is the raw, verbatim thing GEPA
produced: one real reviser step from run 0 — the PARENT it read, the operator + rationale it chose,
and the FULL rubric JSON it emitted — followed by the complete final rubric.

In [6]:
def show_version(vf, label):
    o, b = load_body(vf)
    lin = o.get("lineage") or {}
    print(f"##### {label}: {os.path.basename(vf).split('__')[0]}  "
          f"op={lin.get('operator')}  parent={lin.get('parent_version')}")
    rat = (o.get("notes") or "").replace("reviser_rationale:", "").strip()
    if rat:
        print(f"reviser rationale: {rat}")
    print(b if isinstance(b, str) else json.dumps(b, indent=2)[:1500])
    print()

R0 = f"{REG}/{FAM}_0"
show_version(f"{R0}/v007__prompt.json", "PARENT (re-seed)")
show_version(f"{R0}/v008__prompt.json", "CHILD (what the reviser emitted)")
print("=" * 80)
print("FINAL rubric v013 — FULL verbatim JSON output:\n")
_, bf = load_body(f"{R0}/v013__prompt.json")
print(json.dumps(bf, indent=2))

##### PARENT (re-seed): v007  op=INIT  parent=None
Every proof must be intelligible to someone with basic undergraduate mathematics.

Scoring guidance: Select proofs that do not presuppose advanced graduate-level background; aim for arguments a competent undergraduate reader can follow. This is a stated operational constraint by Aigner & Ziegler.

##### CHILD (what the reviser emitted): v008  op=MECHANIZE  parent=v007
reviser rationale: This revised rubric replaces judgment language with explicit step-by-step checks that a weaker reader can follow mechanically, making it more reliable and consistent.
{
  "name": "Automated Evaluator Rubric for Competitive Programming Solutions",
  "description": "Evaluate the clarity and correctness of a competitive programming solution's proof",
  "criteria": [
    {
      "name": "Intelligibility",
      "description": "Is the proof intelligible to someone with basic undergraduate mathematics?",
      "checks": [
        "Does the proof begin with a 

## 3. Ω — the atomic units mined from all evolutions

Ω is the **union of atomic units across the whole lineage** (not any single rubric). Two honest
points manual inspection forced:

- **Schema fragility.** The original harvester only scanned `criteria[]`/`checks[]` lists, so it
  *silently dropped* runs 1 & 2 (which use named-dimension dicts). Below we run **both** the
  criteria-only extractor (what was actually scored downstream → 6 units) and a **schema-robust**
  extractor (→ 12 units). The gap is the undercount.
- **A lost facet.** Even the robust Ω contains `legibility / clarity / correctness / validity` but
  **not the "elegance / non-ugliness"** facet that run 2 was *seeded* with — GEPA evolved it away.
  So Ω here under-represents the very thing *Proofs from the Book* is about. Keep this in mind when
  reading the bound: it bounds recoverability **relative to the Ω that survived evolution.**

In [7]:
_DIAG = re.compile(r"counterfactual|invariance|delta=|moved the score|edited excerpt|\bMISS\b", re.I)
_JUNK = ("worked_examples", "scores", "scoring", "examples", "score_levels", "levels", "anchors",
         "aggregation", "invariances")


def extract_criteria_only(obj, out):                       # the ORIGINAL extractor (schema A only)
    if isinstance(obj, dict):
        for key in ("criteria", "checks"):
            lst = obj.get(key)
            if isinstance(lst, list):
                for c in lst:
                    if isinstance(c, dict):
                        n, d = c.get("name", ""), c.get("description", "")
                        if d and len(d) > 12 and not _DIAG.search(f"{n} {d}"):
                            out.append(f"{n}: {d}".strip(": ").strip() if n else d)
                        extract_criteria_only(c, out)
        for k, v in obj.items():
            if k not in _JUNK and k not in ("criteria", "checks") and isinstance(v, (dict, list)):
                extract_criteria_only(v, out)
    elif isinstance(obj, list):
        for x in obj:
            extract_criteria_only(x, out)


def extract_robust(obj, out):                              # schema A (lists) + schema B (named dims)
    if isinstance(obj, dict):
        for key in ("criteria", "checks"):
            lst = obj.get(key)
            if isinstance(lst, list):
                for c in lst:
                    if isinstance(c, dict):
                        n, d = c.get("name", ""), c.get("description", "")
                        if d and len(d) > 12 and not _DIAG.search(f"{n} {d}"):
                            out.append(f"{n}: {d}".strip(": ").strip() if n else d)
                        extract_robust(c, out)
        for k, v in obj.items():
            if k in _JUNK or k in ("criteria", "checks"):
                continue
            if isinstance(v, dict) and isinstance(v.get("description"), str) and len(v["description"]) > 12:
                out.append(f"{k}: {v['description']}")
            if isinstance(v, (dict, list)):
                extract_robust(v, out)
    elif isinstance(obj, list):
        for x in obj:
            extract_robust(x, out)


def dedup(cs):
    seen, out = set(), []
    for c in cs:
        k = " ".join(re.sub(r"[^a-z0-9 ]", "", c.lower()).split()[:8])
        if k and k not in seen:
            seen.add(k); out.append(c)
    return out


raw_only, raw_rob = [], []
for vf in sorted(glob.glob(f"{REG}/{FAM}_*/v*__prompt.json")):
    _, b = load_body(vf)
    if isinstance(b, (dict, list)):
        extract_criteria_only(b, raw_only)
        extract_robust(b, raw_rob)
omega_only, omega_rob = dedup(raw_only), dedup(raw_rob)

print(f"Ω criteria-only (what was SCORED downstream): {len(omega_only)} units")
for i, c in enumerate(omega_only):
    print(f"   e{i:02d}: {c[:104]}")
print(f"\nΩ schema-robust (true union across all 3 runs): {len(omega_rob)} units")
for i, c in enumerate(omega_rob):
    tag = "  <-- dropped by criteria-only extractor" if c not in omega_only else ""
    print(f"   e{i:02d}: {c[:88]}{tag}")

Ω criteria-only (what was SCORED downstream): 6 units
   e00: Intelligibility: Is the proof intelligible to someone with basic undergraduate mathematics?
   e01: Validity: Does the proof accurately address the problem and reach a correct conclusion?
   e02: Mathematical Language: Is the mathematical language used in the proof correct and clear?
   e03: Clarity of Argument: Is the argument presented in a clear and logical manner?
   e04: Graduate-Level Background: Does the proof presuppose advanced graduate-level background?
   e05: Background Assumptions: Does the proof assume any background knowledge beyond basic undergraduate mathem

Ω schema-robust (true union across all 3 runs): 12 units
   e00: Intelligibility: Is the proof intelligible to someone with basic undergraduate mathemati
   e01: Validity: Does the proof accurately address the problem and reach a correct conclusion?
   e02: Mathematical Language: Is the mathematical language used in the proof correct and clear?
   e03: C

## 4. How Ω and M were scored — the real input→label data

Before the math, see the **actual data**. The corpus is **math.StackExchange** answers
(`datasets/math/stackexchange/math_se_modeling.csv.gz`), columns `text` and `judgement`.

**We deliberately do NOT use `judgement`** (the dataset's own up/down label) — the whole point is
*unsupervised* recovery, so the target $M$ is a **fresh LLM holistic-quality verdict**, not a human
label. Every signal is produced by the **same one-shot template**, applied by a weak judge
(Llama-3.1-8B), read as a continuous `P(YES)` from the logprobs, then **median-split** to a balanced
binary:

```
_YESNO = "{rubric}\n\nDoes the following item satisfy the criterion above? "
         "Answer with exactly one word: YES or NO.\n\nITEM:\n{text}"
```

- **The target M** plugs the *holistic* question in as `{rubric}`:
  `"Overall, is this answer high quality — the kind a strong expert reviewer would rate in the top
  tier? Answer YES only for clearly strong examples."`
- **Each Ω criterion $X_i$** plugs *that criterion's text* in as `{rubric}` and scores the same item.

So "running Ω" = for each mined criterion $e_i$, run the weak judge over all N answers with $e_i$ as the
rubric → a column of `P(YES)` → $X_i$. The cell below shows real items with their $M$ label and all six
$X_i$ labels, plus how strongly each criterion tracks $M$.

In [8]:
import pandas as pd
ex = pd.read_csv(f"{ROOT}/scored_examples.csv")
legend = dict(l.split("\t", 1) for l in open(f"{ROOT}/criteria_legend.txt").read().splitlines() if "\t" in l)
KX = [c for c in ex.columns if c.startswith("X") and c.endswith("_cont")]
M_PROMPT = ("Overall, is this answer high quality — the kind a strong expert reviewer would rate in "
            "the top tier? Answer YES only for clearly strong examples.")
print(f"corpus rows scored: {len(ex)} | M base-rate (median-split) = {ex['M_bit'].mean():.2f}\n")
for idx in [0, 7]:
    r = ex.iloc[idx]
    print("=" * 92)
    print(f"INPUT  (item {idx}):  {r['text'][:260].replace(chr(10),' ')} ...")
    print(f"\n  TARGET M  <- prompt: \"{M_PROMPT[:70]}...\"")
    print(f"            P(YES)={r['M_cont_holistic']:.3f}  ->  M_bit={int(r['M_bit'])}"
          f"   (dataset's own judgement, UNUSED: {r['dataset_judgement']})")
    print(f"\n  Ω criteria  <- each scored with the SAME template, criterion as the rubric:")
    for c in KX:
        i = c[:-5]
        print(f"     {i}  P(YES)={r[c]:.3f} -> bit={int(r[i+'_bit'])}  | {legend.get(i,'')[:60]}")
print("=" * 92)
print("\nHow strongly each criterion tracks M across all", len(ex), "items (point-biserial corr):")
for c in KX:
    i = c[:-5]
    print(f"  {i}: corr(P(YES), M_holistic) = {ex[c].corr(ex['M_cont_holistic']):+.3f}  | {legend.get(i,'')[:54]}")

corpus rows scored: 250 | M base-rate (median-split) = 0.50

INPUT  (item 0):  Question: Formula for $\sqrt{a-\sqrt{a+\sqrt{a+\ldots}}}$  Supposedly, the infinitely nested radical $$\sqrt{a-\sqrt{a+\sqrt{a+\ldots}}}\tag1\label{1}$$ converges to $$\frac {A-1}{6}+\frac 23\sqrt{4a+A}\sin\left(\frac 13\arctan\frac {2A+1}{3\sqrt 3}\right)\tag ...

  TARGET M  <- prompt: "Overall, is this answer high quality — the kind a strong expert review..."
            P(YES)=0.563  ->  M_bit=1   (dataset's own judgement, UNUSED: 1)

  Ω criteria  <- each scored with the SAME template, criterion as the rubric:
     X0  P(YES)=0.566 -> bit=1  | Intelligibility: Is the proof intelligible to someone with b
     X1  P(YES)=0.732 -> bit=1  | Validity: Does the proof accurately address the problem and 
     X2  P(YES)=0.820 -> bit=1  | Mathematical Language: Is the mathematical language used in 
     X3  P(YES)=0.654 -> bit=0  | Clarity of Argument: Is the argument presented in a clear an
     X4  P(YES)=0.44

### GEPA's own data→label (what the optimizer scored *during* evolution)

GEPA never sees gold labels either — it scores candidates on **fidelity** (reconstruction,
counterfactual, reliability, consistency). The evolved rubric carries two by-products of that loop:
its **score anchors** (what 0.0 / 0.5 / 1.0 mean) and **worked examples** auto-derived from the
*counterfactual* and *invariance* failures it hit. Below are the real ones from v013.

In [9]:
_, b13 = load_body(f"{REG}/{FAM}_0/v013__prompt.json")
print("SCORE ANCHORS (v013):")
for k, v in (b13.get("scores") or {}).items():
    print(f"  {k}: {v[:96]}")
print("\nWORKED EXAMPLES auto-generated from GEPA's measured failures (v013):")
for w in (b13.get("worked_examples") or [])[:4]:
    if isinstance(w, dict):
        print(f"  - [{w.get('problem','')[:38]}]  {str(w.get('solution',''))[:90]}")

SCORE ANCHORS (v013):
  0.0: The proof is completely unclear or incorrect, and the solution is not supported by the given inf
  0.5: The proof is partially unclear or incorrect, or the solution is not fully supported by the given
  1.0: The proof is clear and correct, and the solution is fully supported by the given information.

WORKED EXAMPLES auto-generated from GEPA's measured failures (v013):
  - [Counterfactual MISS [t_up]]  A_k(R) is a polynomial with degree 3k that takes positive values over (0,1) and has a root
  - [Counterfactual MISS [t_down]]  A_k is a polynomial in R with degree 3k that takes positive values over (0,1) and has a ro
  - [Invariance VIOLATION]  A blank_lines change should not affect the score
  - [Counterfactual MISS [t_up]]  The fraction field of R[[x]] is the set of all power series in R[x] with a non-zero denomi


## 5. The math — bounding how recoverable this metric is

**The question.** With no ground-truth label, how much of the holistic judgment $M$ can be
*reconstructed* by articulating the metric into a rubric over Ω and re-executing it? We never compare
to a label $Y$ or a strong-LLM anchor — we measure **recovery through an articulate→re-execute
bottleneck**.

**The objects (all on the same N items):**

- $M$ — the holistic verdict (here: "is this proof top-tier?", median-split to a balanced binary).
- $X_i$ — the per-criterion signal: the executor applying criterion $e_i \in \Omega$ **alone**.
- $\hat M_S$ — the executor's **single** verdict when given the subset $S\subseteq\Omega$ of criteria.

**Two recovery functionals (different $f$, never subtracted):**

$$\check I(S) = I(M; X_S)\quad\text{(Shannon bits — the criteria's *content*, lossless full vector)}$$
$$R(S) = I_{\mathrm{TVD}}(M; \hat M_S) = 2\pi(1-\pi)\,\big|\bar p_1 - \bar p_0\big|\quad\text{(TVD — what one verdict *delivers*)}$$

**The cap.** $\hat M_S$ is a **binary** verdict, so $K_{\text{verdict}}=2$ and
$\mathrm{cap}_{\mathrm{TVD}} = 1 - 1/\min(N, K_{\text{verdict}}) = 1 - 1/2 = \tfrac12$. Every $R$ is
$\le \tfrac12$ **by construction** — the ceiling is set by verdict granularity, not $N$.

**The bound has three layers:**

1. **Discovery** — Ω = the atomic units GEPA actually found (§3).
2. **Selection** — among all $2^{|\Omega|}$ rubrics (subsets), $\mathrm{OPT}_\Omega = \max_S R(S)$.
   We **brute-force the entire lattice**, so $\mathrm{OPT}_\Omega$ is *exact* — the global within-class
   optimum, not a heuristic. Submodularity gives a tractable certificate ($\text{greedy}\ge(1-1/e)\mathrm{OPT}$)
   that we can *check* against the brute force.
3. **Cap** — $\mathrm{OPT}_\Omega \le \mathrm{cap}_{\mathrm{TVD}} = \tfrac12$.

So: $\;R(\text{any rubric over }\Omega) \;\le\; \mathrm{OPT}_\Omega^{\text{exec}} \;\le\; \tfrac12.$

The cells below compute each layer from the real arrays.

In [10]:
# ---------- estimators, inlined so you can read them (match vinfo / submod_conditional) ----------
def tvd_recovery(mhat, M):
    """I_TVD(M; m_hat) = 2 pi (1-pi) |mean(mhat|M=1) - mean(mhat|M=0)|.  mhat: continuous verdict."""
    M = np.asarray(M, float); p = np.asarray(mhat, float)
    keep = np.isfinite(M) & np.isfinite(p); M, p = M[keep], p[keep]
    pi = M.mean()
    return 2 * pi * (1 - pi) * abs(p[M == 1].mean() - p[M == 0].mean())


def _code(X):
    out = np.zeros(len(X), np.int64)
    for j in range(X.shape[1]):
        out = out * 2 + X[:, j].astype(int)
    return out


def _H(counts):
    p = counts[counts > 0].astype(float); p /= p.sum()
    return float(-(p * np.log2(p)).sum())


def I_shannon(M, Xsub):
    """Plug-in Shannon I(M; X_sub) in bits (full joint over the binary columns)."""
    if Xsub.shape[1] == 0:
        return 0.0
    M = M.astype(int); c = _code(Xsub)
    return _H(np.bincount(M)) + _H(np.bincount(c)) - _H(np.bincount(M * (c.max() + 1) + c))


def tvd_mi_joint(M, Xbin):
    """I_TVD(M; X_sub) = 1/2 sum |P(m,x) - P(m)P(x)|, full joint (the lossless-aggregation ceiling)."""
    M = M.astype(int); code = _code(Xbin); tvd = 0.0
    for m in (0, 1):
        pm = (M == m).mean()
        for x in np.unique(code):
            tvd += abs(((M == m) & (code == x)).mean() - pm * (code == x).mean())
    return 0.5 * tvd


def median_split(v):
    v = np.asarray(v, float); thr = np.median(v); out = (v > thr).astype(int)
    return out if 0 < out.sum() < len(out) else (v >= thr).astype(int)


# ---------- load the real scored arrays for aigner ----------
zg = np.load(f"{NPZ}/gamma_criteria_aigner_ziegler_proofs_from_the_book.npz", allow_pickle=True)
zb = np.load(f"{NPZ}/bfc_aigner_ziegler_proofs_from_the_book.npz", allow_pickle=True)
M, X, crits = zg["M"].astype(int), zg["X"].astype(int), list(zg["crits"])
K, N = X.shape[1], len(M)
print(f"Scored: N={N} proofs, K={K} criteria (the 6 that survived into the scoring run).")
print(f"M base-rate = {M.mean():.3f}  (median-split -> balanced binary)\n")
for i, c in enumerate(crits):
    print(f"  X_{i}: {c[:88]}")

Scored: N=250 proofs, K=6 criteria (the 6 that survived into the scoring run).
M base-rate = 0.500  (median-split -> balanced binary)

  X_0: Intelligibility: Is the proof intelligible to someone with basic undergraduate mathemati
  X_1: Validity: Does the proof accurately address the problem and reach a correct conclusion?
  X_2: Mathematical Language: Is the mathematical language used in the proof correct and clear?
  X_3: Clarity of Argument: Is the argument presented in a clear and logical manner?
  X_4: Graduate-Level Background: Does the proof presuppose advanced graduate-level background?
  X_5: Background Assumptions: Does the proof assume any background knowledge beyond basic unde


### 5a. The cap (recovery is bounded by ½ before we compute anything)

In [11]:
print("Verdict is binary  ->  K_verdict = 2")
print("cap_TVD = 1 - 1/min(N, K_verdict) = 1 - 1/2 =", 1 - 1/min(N, 2))
print("Every R(S) below is <= 0.5 by construction.")

Verdict is binary  ->  K_verdict = 2
cap_TVD = 1 - 1/min(N, K_verdict) = 1 - 1/2 = 0.5
Every R(S) below is <= 0.5 by construction.


### 5b. Ideal content $\check I(S)=I(M;X_S)$ and the submodularity certificate

This is the **lossless** ceiling — what the criterion *signals* jointly contain about $M$, with no
single-verdict constraint. We also check the selection certificate: brute-forced $\mathrm{OPT}_\Omega$
vs greedy (should be $\ge 0.632\cdot\mathrm{OPT}$ if $f$ is submodular), and the exact submodularity
ratio $\gamma$ ($\le 1 \Rightarrow$ diminishing returns).

In [12]:
solo = [I_shannon(M, X[:, [i]]) for i in range(K)]
for i in range(K):
    print(f"  solo I(M;X_{i}) = {solo[i]:.3f} bits   | {crits[i][:58]}")
budget = max(2, K // 2)


def opt_brute_shannon(budget):
    best, bestS = 0.0, ()
    for r in range(1, budget + 1):
        for S in itertools.combinations(range(K), r):
            v = I_shannon(M, X[:, list(S)])
            if v > best:
                best, bestS = v, S
    return list(bestS), best


def greedy_shannon(budget):
    S = []
    for _ in range(budget):
        gains = [(I_shannon(M, X[:, S + [e]]) - I_shannon(M, X[:, S]), e) for e in range(K) if e not in S]
        g, e = max(gains)
        if g <= 1e-9:
            break
        S.append(e)
    return S, I_shannon(M, X[:, S])


So, fo = opt_brute_shannon(budget)
Sg, fg = greedy_shannon(budget)
print(f"\n  OPT_O (|S|<={budget}, brute force) = {fo:.3f} bits  at S={So}")
print(f"  greedy                        = {fg:.3f} bits  at S={Sg}")
print(f"  greedy / OPT = {fg/fo if fo>1e-9 else float('nan'):.3f}   (submodular floor 1-1/e = 0.632)")


def gamma_exact():
    fc = {}
    def f(s):
        k = tuple(sorted(s))
        if k not in fc:
            fc[k] = I_shannon(M, X[:, list(k)]) if k else 0.0
        return fc[k]
    g = np.inf
    for r in range(K):
        for S in itertools.combinations(range(K), r):
            rest = [e for e in range(K) if e not in S]
            for ro in range(1, len(rest) + 1):
                for Om in itertools.combinations(rest, ro):
                    den = f(set(S) | set(Om)) - f(S)
                    if den > 1e-6:
                        g = min(g, sum(f(set(S) | {e}) - f(S) for e in Om) / den)
    return float(min(1.0, g)) if np.isfinite(g) else float("nan")


corr = np.nan_to_num(np.corrcoef(X.astype(float).T))
gspec = float(max(0.0, min(1.0, np.linalg.eigvalsh(corr)[0])))
print(f"  exact gamma (brute, K={K}) = {gamma_exact():.3f}   |  spectral-gamma lower bound = {gspec:.3f}")

  solo I(M;X_0) = 0.145 bits   | Intelligibility: Is the proof intelligible to someone with
  solo I(M;X_1) = 0.179 bits   | Validity: Does the proof accurately address the problem an
  solo I(M;X_2) = 0.167 bits   | Mathematical Language: Is the mathematical language used i
  solo I(M;X_3) = 0.278 bits   | Clarity of Argument: Is the argument presented in a clear 
  solo I(M;X_4) = 0.064 bits   | Graduate-Level Background: Does the proof presuppose advan
  solo I(M;X_5) = 0.017 bits   | Background Assumptions: Does the proof assume any backgrou

  OPT_O (|S|<=3, brute force) = 0.340 bits  at S=[0, 3, 5]
  greedy                        = 0.340 bits  at S=[3, 0, 5]
  greedy / OPT = 1.000   (submodular floor 1-1/e = 0.632)
  exact gamma (brute, K=6) = 0.419   |  spectral-gamma lower bound = 0.290


### 5c. Executor recovery $R(S)$ — the EXACT within-class optimum, and PRUNE

`bfc` re-ran the executor on **every** subset of the criteria (the whole $2^K-1$ lattice), so
$\mathrm{OPT}_\Omega^{\text{exec}}$ is exact. The key phenomenon: the best subset is a **strict
subset** — adding all criteria *lowers* $R$ (the single verdict can't juggle them). That non-monotonicity
is **PRUNE**.

In [13]:
mhat = zb["mhat"]; cb = list(zb["crits"]); Kb = len(cb); Mb = zb["M"].astype(int)
keys = [tuple(int(x) for x in k.split(",")) if k else () for k in zb["subset_keys"]]
row = {k: i for i, k in enumerate(keys)}
R = {k: tvd_recovery(mhat[row[k]], Mb) for k in keys}
full = tuple(range(Kb))
best = max(R, key=lambda s: R[s])
print(f"  per-criterion solo R(X_i): {[round(R[(i,)],3) for i in range(Kb)]}")
print(f"  OPT_O^exec (any size, brute) = {R[best]:.3f}  at S={list(best)} (|S|={len(best)})")
print(f"  full-set R(all {Kb})          = {R[full]:.3f}")
print(f"  PRUNE?  best is a strict subset and adding all LOWERS R: "
      f"{len(best) < Kb and R[best] > R[full] + 0.005}")
print(f"\n  -> best rubric DROPS {Kb-len(best)} of {Kb} criteria; full set is {R[full]-R[best]:+.3f} vs best.")

  per-criterion solo R(X_i): [0.075, 0.113, 0.121, 0.114, 0.042, 0.032]
  OPT_O^exec (any size, brute) = 0.133  at S=[1, 2, 3, 5] (|S|=4)
  full-set R(all 6)          = 0.096
  PRUNE?  best is a strict subset and adding all LOWERS R: True

  -> best rubric DROPS 2 of 6 criteria; full set is -0.037 vs best.


### 5d. The aggregation ladder — is the loss the *criteria* or the *single-verdict combiner*?

Hold Ω fixed; change only **who aggregates**. On the same subset:

- $\check I_{\mathrm{TVD}}(M;X_S)$ — full joint (lossless ceiling).
- $R_B$ — combine the per-criterion solo verdicts **outside** the LLM (out-of-fold logistic regression).
- $R_A$ — the LLM's single holistic verdict (= `bfc`).

$\check I \ge R_B \ge R_A$ would mean: collapsing to one scalar costs $\check I - R_B$ (intrinsic), and
the LLM being a worse combiner than arithmetic costs $R_B - R_A$ (fixable plumbing). For aigner the
external combiner **beats** the holistic verdict on the full set — most of PRUNE is plumbing.

In [14]:
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import StratifiedKFold

Xsolo = np.column_stack([mhat[row[(i,)]] for i in range(Kb)])
Xbin = np.column_stack([median_split(Xsolo[:, i]) for i in range(Kb)])


def oof_lr(cols):
    if cols.shape[1] == 0:
        return np.full(len(Mb), Mb.mean())
    oof = np.full(len(Mb), np.nan)
    for tr, te in StratifiedKFold(5, shuffle=True, random_state=0).split(cols, Mb):
        lr = LogisticRegression(C=1.0, max_iter=2000).fit(cols[tr], Mb[tr])
        oof[te] = lr.predict_proba(cols[te])[:, 1]
    return oof


print(f"{'subset':9s} | {'|S|':>3s} | {'Iceil':>7s}{'R_B(LR)':>9s}{'R_A(LLM)':>9s} | "
      f"{'compress':>9s}{'plumbing':>9s}")
for label, S in [("FULL", full), ("LLM-best", best)]:
    Si = list(S)
    Ice = tvd_mi_joint(Mb, Xbin[:, Si]); RB = tvd_recovery(oof_lr(Xsolo[:, Si]), Mb); RA = R[S]
    print(f"{label:9s} | {len(Si):>3d} | {Ice:>7.3f}{RB:>9.3f}{RA:>9.3f} | "
          f"{Ice-RB:>+9.3f}{RB-RA:>+9.3f}")

subset    | |S| |   Iceil  R_B(LR) R_A(LLM) |  compress plumbing
FULL      |   6 |   0.320    0.147    0.096 |    +0.173   +0.051
LLM-best  |   4 |   0.315    0.144    0.133 |    +0.171   +0.011


## 6. The upper bound, stated — and what kind of bound it is

Putting the layers together for this metric (numbers from the cells above):

$$R(\text{any rubric over }\Omega,\ \text{this executor}) \;\le\; \mathrm{OPT}_\Omega^{\text{exec}}
\;\approx\; 0.13 \;\le\; \mathrm{cap}_{\mathrm{TVD}} = \tfrac12.$$

with the lossless content ceiling $\check I(\mathrm{OPT}) \approx 0.34$ bits and the best external
single-verdict $R_B \approx 0.15$.

**Read it honestly — this is a *relative* upper bound, not an absolute one:**

- **Relative to Ω.** It bounds recovery over the units GEPA *discovered and kept*. §3 showed Ω
  under-counts (schema bug → 6 vs 12) and, more importantly, **lost the elegance facet** the metric is
  really about. A richer/better Ω could raise the ceiling. The bound is only as good as Ω.
- **Relative to the executor.** $\mathrm{OPT}_\Omega^{\text{exec}}$ uses one Llama-3.1-8B holistic
  verdict. §5d showed a *better aggregator* (external LR) already beats it — so this is the bound **for
  this executor**, not a law of nature. The genuinely hard ceiling is the single-scalar cap (½) and the
  joint content $\check I$.
- **What's trustworthy.** $\mathrm{OPT}_\Omega^{\text{exec}}$ (brute-forced, exact), the per-criterion
  $I$, the greedy/OPT certificate, the cap, and the $\check I \ge R_B \ge R_A$ ladder. **Not** trusted:
  the exact joint $\gamma$ at this $N$ (under-sampled; spectral-$\gamma$ is only a loose lower bound).

**To validate further yourself:** swap `FAM`/`NPZ` to another metric, widen Ω with the robust extractor
and re-score, or replace the executor in 4d and watch the ladder move.

## 7. Full prompt audit — every LLM prompt the pipeline issues (verbatim, live from source)

Before launching across all metrics, this is the **complete prompt surface**, extracted **live via `ast`
from the actual source files** (not transcribed — what you read here is exactly what runs). Two groups:

- **GEPA optimizer** (`optimizer.py`): the failure-attribution, rubric-revision, and code-revision prompts.
- **GEPA fidelity scorecard** (`measures.py`): reconstruction, rule-grading, counterfactual-edit, oracle.
- **Scoring** (`batch_scoring.py`, `real_gamma.py`): the YES/NO criterion-scoring + rubric-decomposition.
- **Recovery pipeline** (`recovery_trial/recovery_prompts.py`): how the open-source judge labels, the
  (blind + data-driven) recoverers, and the similarity judge.

If any prompt below is wrong, fix it at the source path shown — the audit re-reads it on next run.

In [15]:
import ast, os, re

# resolve repo root (the dir containing methods/metric_implementer)
def _find_repo():
    here = os.path.abspath(".")
    for _ in range(6):
        if os.path.isdir(os.path.join(here, "methods", "metric_implementer")):
            return here
        here = os.path.dirname(here)
    return os.path.abspath("..")
REPO = _find_repo()

TARGETS = [
    ("GEPA optimizer",  "methods/metric_implementer/optimizer.py"),
    ("GEPA scorecard",  "methods/metric_implementer/measures.py"),
    ("Scoring",         "methods/metric_implementer/batch_scoring.py"),
    ("Decomposition",   "methods/metric_implementer/experiments/real_gamma.py"),
    ("Recovery pipeline","methods/metric_implementer/experiments/recovery_trial/recovery_prompts.py"),
]
NAME_RE = re.compile(r"PROMPT|REVISE|_SYS$|YESNO|DECOMP|SUFFIX|RUBRIC|ORACLE|RECON|GRADE|_CF_|TEMPLATE|SCORING|RECOVERER|SIMILARITY")

def prompt_consts(path):
    out = []
    tree = ast.parse(open(path).read())
    for node in tree.body:
        if not isinstance(node, ast.Assign) or len(node.targets) != 1:
            continue
        tgt = node.targets[0]
        name = getattr(tgt, "id", None)
        if not name or not NAME_RE.search(name):
            continue
        try:
            val = ast.literal_eval(node.value)   # handles implicit/paren concatenation
        except Exception:
            continue
        if isinstance(val, str) and len(val) > 30:
            out.append((name, val))
    return out

total = 0
for group, rel in TARGETS:
    path = os.path.join(REPO, rel)
    if not os.path.exists(path):
        print(f"\n[skip] {rel} not found"); continue
    consts = prompt_consts(path)
    for name, val in consts:
        total += 1
        print("=" * 100)
        print(f"[{group}]  {name}   ({rel})")
        print("-" * 100)
        print(val)
print("\n" + "=" * 100)
print(f"TOTAL prompts audited: {total}")

[GEPA optimizer]  _ATTRIBUTE_PROMPT   (methods/metric_implementer/optimizer.py)
----------------------------------------------------------------------------------------------------
A rubric is executed by a WEAKER model than you. For each failure below, decide: could a careful but limited reader score it correctly from the rubric alone (then the failure is the rubric's fault: AMBIGUOUS_PROMPT), or does correct scoring require judgment the rubric cannot transmit to a weak reader as written (JUDGE_LIMITATION)?

RUBRIC:
{body}

FAILURES:
{failures}

Respond ONLY with JSON: {{"attributions": ["AMBIGUOUS_PROMPT"|"JUDGE_LIMITATION", ...] (one per failure, in order)}}
[GEPA optimizer]  _REVISE_PROMPT_KIND   (methods/metric_implementer/optimizer.py)
----------------------------------------------------------------------------------------------------
You are improving the RUBRIC of an automated evaluator (an LLM judge that scores one competitive-programming solution at a time in [0,1]).

CURRENT